In [ ]:
!pip install owlready2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 37.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for owlready2: filename=owlready2-0.50-cp312-cp312-linux_x86_64.whl size=24567083 sha256=7d0fc587349bcb46e78f8f59d9f77bc113c4d101bbcb3600c1208c8584c03ce0
  Stored in directory: /root/.cache/pip/wheels/fb/9d/9d/24582de3856fe0ff897408eb6ed1baa6cd5b7b112a2b9c711c
Successfully built owlready2


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving new_ontology_final.rdf to new_ontology_final.rdf


In [ ]:
from owlready2 import *

# 1. Load the ontology
onto = get_ontology("my_corrected_ontology.rdf").load()

# 2. Get the specific objects using their IDs
with onto:
    Recipe = onto.search_one(iri="*RCH1qyXliabbAWgdLA03caY")

    # Get the Property object explicitly
    hasIngredient_prop = onto.search_one(iri="*R8DfrKZO3NBBj8ScnpEdFMT")

    # Load your helper classes if they exist in the file
    VeganRecipe = onto.search_one(iri="*RVeganRecipe")
    ContainsNutsRecipe = onto.search_one(iri="*RBpTunW4TPQKrAD5P5zeZoF")

# 3. Updated function
def check_recipe_suitability(recipe_name, ingredient_instances):
    with onto:
        print(f"\n--- Processing {recipe_name} ---")

        # Create recipe
        new_recipe = Recipe(recipe_name)

        # Use the Property object explicitly: prop[individual].append(value)
        for ing in ingredient_instances:
            hasIngredient_prop[new_recipe].append(ing)

        sync_reasoner_hermit()

        # Check classification
        if VeganRecipe and new_recipe in VeganRecipe.instances():
            print("Status: ✅ This is a Vegan recipe.")
        else:
            print("Status: ❌ Not a Vegan recipe.")

        if ContainsNutsRecipe and new_recipe in ContainsNutsRecipe.instances():
            print("Status: ❌ ALERT: This recipe contains nuts.")

# 4. Run the test
# Ensure this matches the ID of your almond instance in your RDF file
almond_instance = onto.search_one(iri="*RAlmond1")
check_recipe_suitability("New_Salad", [almond_instance])


--- Processing New_Salad ---


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmp5besfoi1


Status: ❌ Not a Vegan recipe.
Status: ❌ ALERT: This recipe contains nuts.


* Owlready2 * HermiT took 0.9498720169067383 seconds
* Owlready * Reparenting my_corrected_ontology.New_Salad: {webprotege.stanford.edu.RCH1qyXliabbAWgdLA03caY} => {webprotege.stanford.edu.RBpTunW4TPQKrAD5P5zeZoF}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [ ]:
# 1. Setup
!pip install owlready2
from owlready2 import *

# 2. Load the RDF file
# Ensure the filename matches exactly what you uploaded
onto = get_ontology("my_corrected_ontology.rdf").load()

# 3. Access your specific classes and properties
with onto:
    Recipe = IRIS["http://webprotege.stanford.edu/RCH1qyXliabbAWgdLA03caY"]
    NotGlutenFree = IRIS["http://webprotege.stanford.edu/RHq7RKbZd91TZlZTQCm0S3"]
    ContainsNuts = IRIS["http://webprotege.stanford.edu/RBpTunW4TPQKrAD5P5zeZoF"]

    # Object Property: hasIngredient
    hasIngredient = IRIS["http://webprotege.stanford.edu/R8DfrKZO3NBBj8ScnpEdFMT"]

    # Ingredient Classes (from your DL model)
    SoySauce = IRIS["http://webprotege.stanford.edu/RDZQ9z8q5Ul1SpOVXF3Yx7b"]
    WheatFlour = IRIS["http://webprotege.stanford.edu/RBrQuOHR0cfyecPXNTOENYB"]

# 4. Integration Function
def filter_recommendation(name, ingredients):
    with onto:
        # Create a new recipe individual
        new_rec = Recipe(name)

        # Add ingredients to the recipe
        for ing_class in ingredients:
            ing_instance = ing_class()
            new_rec.R8DfrKZO3NBBj8ScnpEdFMT.append(ing_instance)

        # Run Reasoner (Transitive Reasoning)
        sync_reasoner_hermit()

        # Check against logic classes
        print(f"\n--- Checking: {name} ---")
        is_safe = True
        if new_rec in NotGlutenFree.instances():
            print("REJECTED: Contains Gluten")
            is_safe = False
        if new_rec in ContainsNuts.instances():
            print("REJECTED: Contains Nuts")
            is_safe = False

        if is_safe:
            print("ACCEPTED: Safe for user diet.")

# 5. Test the integration
# Simulate a DL recommendation containing Soy Sauce and Wheat Flour
filter_recommendation("DL_Ramen_Suggestion", [SoySauce, WheatFlour])

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpk506l2ki



--- Checking: DL_Ramen_Suggestion ---
REJECTED: Contains Gluten


* Owlready2 * HermiT took 1.4942045211791992 seconds
* Owlready * Reparenting webprotege.stanford.edu.RHq7RKbZd91TZlZTQCm0S3: {owl.Thing} => {webprotege.stanford.edu.RCH1qyXliabbAWgdLA03caY}
* Owlready * Reparenting webprotege.stanford.edu.RBpTunW4TPQKrAD5P5zeZoF: {owl.Thing} => {webprotege.stanford.edu.RCH1qyXliabbAWgdLA03caY}
* Owlready * Reparenting my_corrected_ontology.DL_Ramen_Suggestion: {webprotege.stanford.edu.RCH1qyXliabbAWgdLA03caY} => {webprotege.stanford.edu.RHq7RKbZd91TZlZTQCm0S3}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
